# Implementation of the example in Appendix C of the thesis of Antoine Dumas:

## Application appui plan et deux goupilles

In [1]:
import math
import sympy as sp
import numpy as np
from typing import List, Tuple
from functools import partial
import sympy as sp
from IPython import display

import otaf

Since the implementation in dumas work is a bit different and that we don't really care about the linerization rn, we have to ceate some intermediary functions for the optimization.

We have multiple sets of variables, and the optimization is only done on the gap varibles (g), meaning we need to have a first function that fixes the other variables (using partial) or something, so that we have in the end only a function that takes as an input the gaps.

### Original values from dumas work (we choose parameter set 1)

In [2]:
L = [100, 40, 30, 30, 20, 20, 120, 50, 40, 50, -30]
d_max = 0.25

mu_d_ext = 20
sigma_d_ext = 0.06

mu_d_int = 19.8
sigma_d_int = 0.06

mu_trans = 0
sigma_trans = 0.01
mu_rot = 0 
sigma_rot = 0.001

st = sigma_trans
sr = sigma_rot

### Let's use the equations in C.15 to calculate the equivalent distribution parameters.

Since all mean shifts are 0, the meansz of the rotational components are also 0

In [3]:
const = L[0]*L[10] - L[1]*L[9] 
sigma_beta_d_0 = np.sqrt(((L[0]/const)*st)**2 +(((L[9]-L[0])/const)*st)**2 +((L[9]/const)*st)**2)
sigma_gamma_d_0 = np.sqrt(((L[1]/const)*st)**2 +(((L[10]-L[1])/const)*st)**2 +((L[10]/const)*st)**2)
sigma_beta_d_1 = np.sqrt(((L[0]/const)*st)**2 +(((L[9]-L[0])/const)*st)**2 +((L[9]/const)*st)**2)
sigma_gamma_d_1 = sigma_gamma_d_0
sigma_beta_d_2 = np.sqrt(2*(st**2))/L[2]
sigma_gamma_d_2 = np.sqrt(2*(st**2))/L[2]
sigma_beta_d_3 = np.sqrt(2*(st**2))/L[4]
sigma_gamma_d_3 = np.sqrt(2*(st**2))/L[4]
sigma_beta_d_4 = np.sqrt(2*(st**2))/L[3]
sigma_gamma_d_4 = np.sqrt(2*(st**2))/L[3]
sigma_beta_d_5 = np.sqrt(2*(st**2))/L[5]
sigma_gamma_d_5 = np.sqrt(2*(st**2))/L[5]

These values are only valid for the parameterization where the position of the two upper and lower circles at the end of the cylindrical features is modeled, not the orientation and position of the cylindrical feature itself. We want on the other hand have parameters to model that, and find and equivalent set of parameters.

More specifically, we need a tolerance and capability value to use as knowledge constraints.

Since we have a feature (a circle), and a set of distribution parameters that controls the diameter and the position of this circle, we can suppose that we have a tolerance zone that is the zone between 2 concentric circles.

We can arbitrarily fix the capability to some values, let's say 1, and find the tolerance value. Under the assumption of a centered normal distribution, a capability of $C_p=1$ implies that the tolerance width is $6 \sigma$. The probability of failure is thus the area under the tails where $|Z|>3$, calculated as $2(1−P(X \leq 3))$, which is equal to $2.699796 \cdot 1e^{-3}$.

Using the parameters sigma_d_ext and sigma_trans and inject it in the formula for the local circular defect (the defect is centered on the nominal), then we can treat this as a traditional case fo having a normal distribution within a tolerance interval $t$. 

We have $\sigma_\delta = t / 6C$, since $C=1$, $t=6*\sigma_\delta$

In [4]:
std_delta_cyl = np.sqrt(sigma_d_ext**2 + sigma_trans**2)
# This is from the formula for the standard deviation of the measured defect for a circular tolerance zone
# So this is our target in optimization
print(std_delta_cyl)
# For thea measured defect on the planar feature the target is sigma_trans (already a measured standard)

0.0608276253029822


In [5]:
def sigma_delta_circular_feature(theta, sr=sigma_d_ext, su=sigma_trans, sv=sigma_trans):
    """This function is used to obtain the standard deviation of the defect for a point in the circular features 
    This can also be used as a basis for a constraint function (to always have the same max standard deviation over the circular feature)
    """
    return np.sqrt(sr**2 + np.cos(theta)**2 * su**2 + np.sin(theta)**2 * sv**2)

def sigma_delta_3D_plane(x,y, sw=sigma_trans, sa=sigma_rot, sb=sigma_rot):
    """This function is used to obtain the standard deviation of the defect for a point on the plane feature 
    This can also be used as a basis for a constraint function
    """
    return np.sqrt(sw**2 + y**2 * sa**2 + x**2 * sb**2)

In [6]:
# Here we define our constraints for the target standard deviations for each feature
# Since we disregard mean shifts We can directly use the expression of the standard deviation 
# of the measured defect
# We must find the max though, at least for the circular feature since for the plane it is on the corners
# We also don't include any correlation yet

This means that the cylindrical feature must be within two cylinders of radius difference of t_eq

We'll also obtain the 

In [7]:
mats = otaf.example_models.model_dumas_cython.build_constraint_matrices(L,32)

In [8]:
SOCAM =otaf.SystemOfConstraintsAssemblyModel(matrices=list(mats))

In [9]:
d_labels = [sp.Symbol(otaf.example_models.model_dumas_cython.x_full_labels_mapping[lab]) for lab in otaf.example_models.model_dumas_cython.x_full_labels]
g_labels = [sp.Symbol(otaf.example_models.model_dumas_cython.g_labels_mapping[lab]) for lab in otaf.example_models.model_dumas_cython.g_labels]
SOCAM.deviation_symbols = d_labels
SOCAM.gap_symbols = g_labels
SOCAM.embedOptimizationVariable()

In [10]:
SOCAM.test_zero_deviation_feasibility()

{'success': True,
 'status': 0,
 'message': 'Optimization terminated successfully. (HiGHS Status 7: Optimal)',
 'gap_values': array([-0., -0., -0., -0., -0., -0., -0., -0., -0., -0., -0., -0., -0.,
        -0., -0., -0., -0., -0.]),
 'objective': 0.0}

In [11]:
d_labels

[u_d_0,
 beta_d_0,
 gamma_d_0,
 u_d_1,
 beta_d_1,
 gamma_d_1,
 v_d_2,
 w_d_2,
 beta_d_2,
 gamma_d_2,
 v_d_3,
 w_d_3,
 beta_d_3,
 gamma_d_3,
 v_d_4,
 w_d_4,
 beta_d_4,
 gamma_d_4,
 v_d_5,
 w_d_5,
 beta_d_5,
 gamma_d_5,
 d_d_2,
 d_d_6,
 d_d_4,
 d_d_7,
 v_d_8,
 w_d_8,
 beta_d_8,
 gamma_d_8]

In [12]:
mu_list = [.0]*22+[mu_d_ext,mu_d_int,mu_d_ext,mu_d_int]+[.0]*4
sigma_list = [st, sigma_beta_d_0, sigma_gamma_d_0, 
              st, sigma_beta_d_1, sigma_gamma_d_1,
              st, st, sigma_beta_d_2, sigma_gamma_d_2,
              st, st, sigma_beta_d_3, sigma_gamma_d_3,
              st, st, sigma_beta_d_4, sigma_gamma_d_4,
              st, st, sigma_beta_d_5, sigma_gamma_d_5,
              sigma_d_ext,sigma_d_int,sigma_d_ext,sigma_d_int,
              st, st, sr, sr]

In [13]:
    # The gap vector is of dimension 16 + 1 (due to the added slack variable)
deviation_symbols = otaf.example_models.model_dumas_cython.x_full_labels
# Here we have a distribution in the standard space. 
defect_distribution = otaf.distribution.get_composed_normal_defect_distribution(
    deviation_symbols,
    mu_list = mu_list,
    sigma_list = sigma_list)
sample_U_50000 = np.array(defect_distribution.getSample(50000))

In [ ]:
def get_compatibility_constraints(x_full, L, d_max):
    return partial(otaf.example_models.model_dumas_cython.constraints_eq_base, x_full=x_full, L=L, d_max=d_max)

def get_interface_constraints(x_full, L, d_max):
    return partial(otaf.example_models.model_dumas_cython.constraints_ineq_slack, x_full=x_full, L=L, d_max=d_max)

def get_optim_functions(x_full, L=L, d_max=d_max):
    return get_compatibility_constraints(x_full, L, d_max), get_interface_constraints(x_full, L, d_max)

def get_opt_problem(x_full, L=L, d_max=d_max):
    compat, interf = get_optim_functions(x_full, L, d_max)
    ineq_cons = {"type":"ineq", 'fun':lambda g: -1*interf(g)} #change sign for slsqp
    eq_cons = {"type":"eq", 'fun':lambda g: compat(g)}
    return eq_cons, ineq_cons
    

### Now lets check if we get the same values than A. Dumas with his different parameter values.